In [2]:
import pandas as pd
import numpy as np
import cupy as cp
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRFRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口
# ==========================================
INPUT_FILE = "P4_Cleaned_Dataset.csv"
OUTPUT_CSV = "P4_Nested_MOBO_Metrics_Fast.csv"
OPTUNA_TRIALS = 50  # [加速策略 1] 迭代次数降至 50，足以逼近帕累托前沿
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def objective(trial, X_df, y_np, groups):
    # 1. 动态生成特征掩码
    active_features = []
    for col in X_df.columns:
        if trial.suggest_categorical(f'mask_{col}', [True, False]):
            active_features.append(col)
            
    if len(active_features) == 0:
        return float('inf'), len(X_df.columns)
        
    # 2. 超参数采样
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200, step=50),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.3, 0.9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'tree_method': 'gpu_hist',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 3. 恢复内部留一交叉验证
    logo = LeaveOneGroupOut()
    mae_scores = [] 
    
    X_np_subset = X_df[active_features].values
    
    for train_idx, val_idx in logo.split(X_np_subset, y_np, groups):
        X_tr, X_val = X_np_subset[train_idx], X_np_subset[val_idx]
        y_tr, y_val = y_np[train_idx], y_np[val_idx]
        
        model = XGBRFRegressor(**params)
        model.fit(cp.array(X_tr), cp.array(y_tr))
        
        preds = cp.asnumpy(model.predict(cp.array(X_val)))
        mae_scores.append(mean_absolute_error(y_val, preds))
        
    return np.mean(mae_scores), len(active_features)

def run_nested_mobo(input_file, output_csv):
    if not os.path.exists(input_file):
        print(f"错误: 找不到文件 {input_file}。请检查路径。")
        return
        
    df = pd.read_csv(input_file)
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    feature_cols = [col for col in df.columns if col not in meta_cols]
    
    years = sorted(df['Year'].unique())
    all_y_true, all_y_pred = [], []
    fold_results = []
    
    print(f"\n{'='*75}")
    print(f">>> 启动 MOBO 联合优化 (修改)")
    print(f"    初始特征维度: {len(feature_cols)}")
    print(f"    单年寻优次数: {OPTUNA_TRIALS}")
    print(f"{'='*75}")

    for test_year in years:
        train_df = df[df['Year'] != test_year]
        test_df = df[df['Year'] == test_year]
        
        X_train_df = train_df[feature_cols]
        y_train_np = train_df['yield'].values
        groups_train = train_df['Year'].values  # 用于内层信息隔离
        
        X_test_df = test_df[feature_cols]
        y_test_np = test_df['yield'].values
        
        print(f"    [Year {test_year}] 正在寻找 Pareto 最优解...", end="", flush=True)
        
        # 声明多目标：分别最小化 MAE 和 特征数量
        study = optuna.create_study(directions=['minimize', 'minimize'], sampler=TPESampler(seed=42))
        func = lambda trial: objective(trial, X_train_df, y_train_np, groups_train)
        study.optimize(func, n_trials=OPTUNA_TRIALS)
        
        # 从 Pareto 前沿中提取最佳试验
        # 策略：以预测精度为绝对第一优先级，选择内层评级 MAE 最低的解
        pareto_front = study.best_trials
        best_trial = sorted(pareto_front, key=lambda t: t.values[0])[0]
        
        # 解析最佳参数和被选中的特征
        best_params = {k: v for k, v in best_trial.params.items() if not k.startswith('mask_')}
        best_params.update({'tree_method': 'gpu_hist', 'random_state': 42, 'n_jobs': -1})
        
        selected_features = [col for col in feature_cols if best_trial.params.get(f'mask_{col}', False)]
        
        print(f" 完成.")
        print(f"      -> 内部评级 MAE: {best_trial.values[0]:.2f} | 选用特征数: {len(selected_features)}/{len(feature_cols)}")
        
        # 提取最终使用的特征矩阵
        X_train_np_final = X_train_df[selected_features].values
        X_test_np_final = X_test_df[selected_features].values
        
        # 最终模型拟合 (使用外层全量训练集)
        final_model = XGBRFRegressor(**best_params)
        final_model.fit(cp.array(X_train_np_final), cp.array(y_train_np))
        
        # 预测与记录
        y_pred = cp.asnumpy(final_model.predict(cp.array(X_test_np_final)))
        
        all_y_true.extend(y_test_np)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test_np, y_pred)
        m['Test_Year'] = str(test_year)
        m['Used_Features'] = len(selected_features)
        m['Best_Params'] = str({k: best_params[k] for k in ['max_depth', 'colsample_bynode', 'subsample']})
        fold_results.append(m)
        
        print(f"      -> [验证结果] RRMSE: {m['RRMSE(%)']:.2f}% | MAPE: {m['MAPE(%)']:.2f}%\n")

    # 全局指标计算 (Overall Pooled)
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall_Pooled'
    global_metrics['Used_Features'] = 'N/A'
    global_metrics['Best_Params'] = 'N/A'
    fold_results.append(global_metrics)
    
    print(f"{'='*75}")
    print(f">>> 联合优化全局汇总")
    m = global_metrics
    print(f"    R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}%")
    print(f"    MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
    print(f"{'='*75}")

    cols = ['Test_Year', 'Used_Features', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    pd.DataFrame(fold_results)[cols].to_csv(output_csv, index=False)
    print(f"指标已保存至: {output_csv}")

if __name__ == "__main__":
    run_nested_mobo(INPUT_FILE, OUTPUT_CSV)


>>> 启动 MOBO 联合优化 (RTX 4050 极速对齐版)
    初始特征维度: 100
    单年寻优次数: 50
    [Year 2016] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 514.51 | 选用特征数: 37/100
      -> [验证结果] RRMSE: 13.43% | MAPE: 8.82%

    [Year 2017] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 518.65 | 选用特征数: 56/100
      -> [验证结果] RRMSE: 13.28% | MAPE: 11.10%

    [Year 2018] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 545.94 | 选用特征数: 44/100
      -> [验证结果] RRMSE: 11.58% | MAPE: 9.07%

    [Year 2019] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 546.05 | 选用特征数: 46/100
      -> [验证结果] RRMSE: 12.12% | MAPE: 9.36%

    [Year 2020] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 533.52 | 选用特征数: 44/100
      -> [验证结果] RRMSE: 11.30% | MAPE: 9.29%

    [Year 2021] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 556.76 | 选用特征数: 38/100
      -> [验证结果] RRMSE: 11.04% | MAPE: 9.01%

>>> 联合优化全局汇总
    R2: 0.345 | RMSE: 750.52 | RRMSE: 12.19%
    MAE: 533.98 | MAPE: 9.44% | d-index: 0.717
指标已保存至: P4_Nested_MOBO_Metrics_Fast.csv


260412 最新的 感觉作废 结果不好
搜索次数：OPTUNA_TRIALS 提升至 50，让算法有充分的机会在特征海中寻找最优解。
放宽了超参数边界：树深度、采样率等参数范围拓宽，赋予模型更强的非线性表达能力以应对特征减少带来的信息损失。
纠正了优化目标（最关键）：内层指引 Optuna 进化的指标已从 MSE 彻底替换为 MAE，使其与你最终关注的绝对百分比误差（MAPE）方向保持一致。

In [1]:
import pandas as pd
import numpy as np
import cupy as cp
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRFRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口
# ==========================================
INPUT_FILE = "P4_Cleaned_Dataset.csv"
OUTPUT_CSV = "P4_Nested_MOBO_Metrics_Fast.csv"
OPTUNA_TRIALS = 50  # [加速策略 1] 迭代次数降至 50，足以逼近帕累托前沿
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def objective(trial, X_df, y_np, groups):
    """
    多目标优化函数：
    目标 1：最小化内部验证的 MAE (与最终 MAPE 目标对齐)
    目标 2：最小化特征使用数量
    """
    # 1. 动态生成特征掩码 (Feature Mask)
    active_features = []
    for col in X_df.columns:
        if trial.suggest_categorical(f'mask_{col}', [True, False]):
            active_features.append(col)
            
    # 若极端情况下剔除了所有特征，返回极大惩罚值
    if len(active_features) == 0:
        return float('inf'), len(X_df.columns)
        
    # 2. 随机森林超参数采样 [加速策略 2: 压缩树模型上限，减轻显存与计算压力]
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150, step=50),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.2, 0.9),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'tree_method': 'gpu_hist',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 3. 内部验证策略 [加速策略 3: 使用按年份分组的单次划分，阻断信息泄露的同时缩减内层计算量]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    mae_scores = [] 
    
    X_np_subset = X_df[active_features].values
    
    for train_idx, val_idx in gss.split(X_np_subset, y_np, groups):
        X_tr, X_val = X_np_subset[train_idx], X_np_subset[val_idx]
        y_tr, y_val = y_np[train_idx], y_np[val_idx]
        
        model = XGBRFRegressor(**params)
        model.fit(cp.array(X_tr), cp.array(y_tr))
        
        preds = cp.asnumpy(model.predict(cp.array(X_val)))
        mae_scores.append(mean_absolute_error(y_val, preds))
        
    return np.mean(mae_scores), len(active_features)

def run_nested_mobo(input_file, output_csv):
    if not os.path.exists(input_file):
        print(f"错误: 找不到文件 {input_file}。请检查路径。")
        return
        
    df = pd.read_csv(input_file)
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    feature_cols = [col for col in df.columns if col not in meta_cols]
    
    years = sorted(df['Year'].unique())
    all_y_true, all_y_pred = [], []
    fold_results = []
    
    print(f"\n{'='*75}")
    print(f">>> 启动 MOBO 联合优化 (RTX 4050 极速对齐版)")
    print(f"    初始特征维度: {len(feature_cols)}")
    print(f"    单年寻优次数: {OPTUNA_TRIALS}")
    print(f"{'='*75}")

    for test_year in years:
        train_df = df[df['Year'] != test_year]
        test_df = df[df['Year'] == test_year]
        
        X_train_df = train_df[feature_cols]
        y_train_np = train_df['yield'].values
        groups_train = train_df['Year'].values  # 用于内层信息隔离
        
        X_test_df = test_df[feature_cols]
        y_test_np = test_df['yield'].values
        
        print(f"    [Year {test_year}] 正在寻找 Pareto 最优解...", end="", flush=True)
        
        # 声明多目标：分别最小化 MAE 和 特征数量
        study = optuna.create_study(directions=['minimize', 'minimize'], sampler=TPESampler(seed=42))
        func = lambda trial: objective(trial, X_train_df, y_train_np, groups_train)
        study.optimize(func, n_trials=OPTUNA_TRIALS)
        
        # 从 Pareto 前沿中提取最佳试验
        # 策略：以预测精度为绝对第一优先级，选择内层评级 MAE 最低的解
        pareto_front = study.best_trials
        best_trial = sorted(pareto_front, key=lambda t: t.values[0])[0]
        
        # 解析最佳参数和被选中的特征
        best_params = {k: v for k, v in best_trial.params.items() if not k.startswith('mask_')}
        best_params.update({'tree_method': 'gpu_hist', 'random_state': 42, 'n_jobs': -1})
        
        selected_features = [col for col in feature_cols if best_trial.params.get(f'mask_{col}', False)]
        
        print(f" 完成.")
        print(f"      -> 内部评级 MAE: {best_trial.values[0]:.2f} | 选用特征数: {len(selected_features)}/{len(feature_cols)}")
        
        # 提取最终使用的特征矩阵
        X_train_np_final = X_train_df[selected_features].values
        X_test_np_final = X_test_df[selected_features].values
        
        # 最终模型拟合 (使用外层全量训练集)
        final_model = XGBRFRegressor(**best_params)
        final_model.fit(cp.array(X_train_np_final), cp.array(y_train_np))
        
        # 预测与记录
        y_pred = cp.asnumpy(final_model.predict(cp.array(X_test_np_final)))
        
        all_y_true.extend(y_test_np)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test_np, y_pred)
        m['Test_Year'] = str(test_year)
        m['Used_Features'] = len(selected_features)
        m['Best_Params'] = str({k: best_params[k] for k in ['max_depth', 'colsample_bynode', 'subsample']})
        fold_results.append(m)
        
        print(f"      -> [验证结果] RRMSE: {m['RRMSE(%)']:.2f}% | MAPE: {m['MAPE(%)']:.2f}%\n")

    # 全局指标计算 (Overall Pooled)
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall_Pooled'
    global_metrics['Used_Features'] = 'N/A'
    global_metrics['Best_Params'] = 'N/A'
    fold_results.append(global_metrics)
    
    print(f"{'='*75}")
    print(f">>> 联合优化全局汇总")
    m = global_metrics
    print(f"    R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}%")
    print(f"    MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
    print(f"{'='*75}")

    cols = ['Test_Year', 'Used_Features', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    pd.DataFrame(fold_results)[cols].to_csv(output_csv, index=False)
    print(f"指标已保存至: {output_csv}")

if __name__ == "__main__":
    run_nested_mobo(INPUT_FILE, OUTPUT_CSV)


>>> 启动 MOBO 联合优化 (RTX 4050 极速对齐版)
    初始特征维度: 100
    单年寻优次数: 50
    [Year 2016] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 491.99 | 选用特征数: 52/100
      -> [验证结果] RRMSE: 13.52% | MAPE: 8.82%

    [Year 2017] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 523.44 | 选用特征数: 51/100
      -> [验证结果] RRMSE: 14.20% | MAPE: 12.05%

    [Year 2018] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 574.44 | 选用特征数: 44/100
      -> [验证结果] RRMSE: 11.61% | MAPE: 9.31%

    [Year 2019] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 576.43 | 选用特征数: 41/100
      -> [验证结果] RRMSE: 14.77% | MAPE: 11.72%

    [Year 2020] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 557.04 | 选用特征数: 45/100
      -> [验证结果] RRMSE: 11.35% | MAPE: 9.47%

    [Year 2021] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 556.28 | 选用特征数: 43/100
      -> [验证结果] RRMSE: 12.86% | MAPE: 9.78%

>>> 联合优化全局汇总
    R2: 0.238 | RMSE: 809.50 | RRMSE: 13.15%
    MAE: 577.25 | MAPE: 10.20% | d-index: 0.662
指标已保存至: P4_Nested_MOBO_Metrics_Fast.csv


MSE+特征数（MOBO）
LOYO（按年分组）

In [1]:
import pandas as pd
import numpy as np
import cupy as cp
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import LeaveOneGroupOut
from xgboost import XGBRFRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口
# ==========================================
INPUT_FILE = "P4_Cleaned_Dataset.csv"
OUTPUT_CSV = "P4_Nested_MOBO_Metrics.csv"
OPTUNA_TRIALS = 40  # 建议保持在 30-50 之间
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def objective(trial, X_df, y_np, groups):
    """
    多目标优化函数：
    目标 1：最小化内部跨年验证的 MSE
    目标 2：最小化特征使用数量
    """
    # 1. 动态生成特征掩码 (Feature Mask)
    active_features = []
    for col in X_df.columns:
        if trial.suggest_categorical(f'mask_{col}', [True, False]):
            active_features.append(col)
            
    # 若极端情况下剔除了所有特征，返回极大惩罚值
    if len(active_features) == 0:
        return float('inf'), len(X_df.columns)
        
    # 2. 随机森林超参数采样
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, step=50),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.3, 0.8),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'tree_method': 'gpu_hist',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 3. 内部留一交叉验证 (按年份分组，阻断信息泄露)
    logo = LeaveOneGroupOut()
    mse_scores = []
    
    X_np_subset = X_df[active_features].values
    
    for train_idx, val_idx in logo.split(X_np_subset, y_np, groups):
        X_tr, X_val = X_np_subset[train_idx], X_np_subset[val_idx]
        y_tr, y_val = y_np[train_idx], y_np[val_idx]
        
        model = XGBRFRegressor(**params)
        model.fit(cp.array(X_tr), cp.array(y_tr))
        
        preds = cp.asnumpy(model.predict(cp.array(X_val)))
        mse_scores.append(mean_squared_error(y_val, preds))
        
    return np.mean(mse_scores), len(active_features)

def run_nested_mobo(input_file, output_csv):
    if not os.path.exists(input_file):
        print(f"文件未找到: {input_file}")
        return
        
    df = pd.read_csv(input_file)
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    feature_cols = [col for col in df.columns if col not in meta_cols]
    
    years = sorted(df['Year'].unique())
    all_y_true, all_y_pred = [], []
    fold_results = []
    
    print(f"\n{'='*75}")
    print(f">>> 启动 MOBO 联合优化 (特征选择 + 超参数寻优)")
    print(f"    初始特征维度: {len(feature_cols)}")
    print(f"{'='*75}")

    for test_year in years:
        train_df = df[df['Year'] != test_year]
        test_df = df[df['Year'] == test_year]
        
        X_train_df = train_df[feature_cols]
        y_train_np = train_df['yield'].values
        groups_train = train_df['Year'].values  # 用于内层分组
        
        X_test_df = test_df[feature_cols]
        y_test_np = test_df['yield'].values
        
        print(f"    [Year {test_year}] 正在寻找 Pareto 最优解...", end="", flush=True)
        
        # 声明多目标：分别最小化误差和特征数
        study = optuna.create_study(directions=['minimize', 'minimize'], sampler=TPESampler(seed=42))
        func = lambda trial: objective(trial, X_train_df, y_train_np, groups_train)
        study.optimize(func, n_trials=OPTUNA_TRIALS)
        
        # 从 Pareto 前沿中提取最佳试验
        # 策略：以预测精度为第一优先级，在 Pareto 前沿中选择 MSE 最低的组合
        pareto_front = study.best_trials
        best_trial = sorted(pareto_front, key=lambda t: t.values[0])[0]
        
        # 解析最佳参数和被选中的特征
        best_params = {k: v for k, v in best_trial.params.items() if not k.startswith('mask_')}
        # best_params.update({'tree_method': 'hist', 'device': 'cuda', 'random_state': 42, 'n_jobs': -1})
        best_params.update({'tree_method': 'gpu_hist', 'random_state': 42, 'n_jobs': -1})
        
        selected_features = [col for col in feature_cols if best_trial.params.get(f'mask_{col}', False)]
        
        print(f" 完成.")
        print(f"      -> 内部评级 MSE: {best_trial.values[0]:.1f} | 选用特征数: {len(selected_features)}/{len(feature_cols)}")
        
        # 提取最终使用的特征矩阵
        X_train_np_final = X_train_df[selected_features].values
        X_test_np_final = X_test_df[selected_features].values
        
        # 最终模型拟合
        final_model = XGBRFRegressor(**best_params)
        final_model.fit(cp.array(X_train_np_final), cp.array(y_train_np))
        
        # 预测与记录
        y_pred = cp.asnumpy(final_model.predict(cp.array(X_test_np_final)))
        
        all_y_true.extend(y_test_np)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test_np, y_pred)
        m['Test_Year'] = str(test_year)
        m['Used_Features'] = len(selected_features)
        m['Best_Params'] = str({k: best_params[k] for k in ['max_depth', 'colsample_bynode', 'subsample']})
        fold_results.append(m)
        
        print(f"      -> [验证结果] RRMSE: {m['RRMSE(%)']:.2f}% | MAPE: {m['MAPE(%)']:.2f}%\n")

    # 全局指标计算
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall_Pooled'
    global_metrics['Used_Features'] = 'N/A'
    global_metrics['Best_Params'] = 'N/A'
    fold_results.append(global_metrics)
    
    print(f"{'='*75}")
    print(f">>> 联合优化全局汇总")
    m = global_metrics
    print(f"    R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}%")
    print(f"    MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
    print(f"{'='*75}")

    cols = ['Test_Year', 'Used_Features', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    pd.DataFrame(fold_results)[cols].to_csv(output_csv, index=False)
    print(f"指标已保存至: {output_csv}")

if __name__ == "__main__":
    run_nested_mobo(INPUT_FILE, OUTPUT_CSV)


>>> 启动 MOBO 联合优化 (特征选择 + 超参数寻优)
    初始特征维度: 100
    [Year 2016] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MSE: 497974.4 | 选用特征数: 44/100
      -> [验证结果] RRMSE: 13.46% | MAPE: 8.79%

    [Year 2017] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MSE: 540281.8 | 选用特征数: 45/100
      -> [验证结果] RRMSE: 14.52% | MAPE: 12.40%

    [Year 2018] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MSE: 611268.3 | 选用特征数: 46/100
      -> [验证结果] RRMSE: 11.55% | MAPE: 8.96%

    [Year 2019] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MSE: 578761.7 | 选用特征数: 38/100
      -> [验证结果] RRMSE: 13.26% | MAPE: 10.37%

    [Year 2020] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MSE: 575500.3 | 选用特征数: 46/100
      -> [验证结果] RRMSE: 11.18% | MAPE: 9.20%

    [Year 2021] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MSE: 605499.1 | 选用特征数: 46/100
      -> [验证结果] RRMSE: 11.62% | MAPE: 9.05%

>>> 联合优化全局汇总
    R2: 0.289 | RMSE: 781.92 | RRMSE: 12.70%
    MAE: 555.15 | MAPE: 9.79% | d-index: 0.691
指标已保存至: P4_Nested_MOBO_Metrics.csv


仅MSE（单目标）
80/20 Hold-out

In [1]:
import pandas as pd
import numpy as np
import cupy as cp
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from xgboost import XGBRFRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# 关闭 Optuna 的逐次试验打印，保持终端输出清爽
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口
# ==========================================
INPUT_FILE = "P4_Cleaned_Dataset.csv"
OUTPUT_CSV = "P4_Nested_Metrics.csv"
# 若修改后速度依然不理想，可将此处改为 15 或 20
OPTUNA_TRIALS = 30  
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def objective(trial, X_train_np, y_train_np):
    """
    加速版 Optuna 内层目标函数：
    使用 80/20 Hold-out 验证代替多折交叉验证，大幅减少评估单组超参数的时间。
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, step=50),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.2, 0.6),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'tree_method': 'gpu_hist',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 将当前的训练集按 80% 训练 / 20% 验证进行单次随机划分
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_np, y_train_np, test_size=0.2, random_state=42
    )
    
    model = XGBRFRegressor(**params)
    
    # 显式 GPU 训练
    model.fit(cp.array(X_tr), cp.array(y_tr))
    
    # 预测并计算 MSE
    preds = cp.asnumpy(model.predict(cp.array(X_val)))
    mse = mean_squared_error(y_val, preds)
        
    return mse

def run_nested_optimization(input_file, output_csv):
    if not os.path.exists(input_file):
        print(f"文件未找到: {input_file}。请先运行数据固化脚本 01_prepare_P4_data.py。")
        return
        
    df = pd.read_csv(input_file)
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    feature_cols = [col for col in df.columns if col not in meta_cols]
    
    years = sorted(df['Year'].unique())
    all_y_true, all_y_pred = [], []
    fold_results = []
    
    print(f"\n{'='*70}")
    print(f">>> 启动嵌套 LOYO 交叉验证与超参数优化 (加速版)")
    print(f"    数据集特征维度: {len(feature_cols)}")
    print(f"    每个外层折叠的 Optuna 寻优次数: {OPTUNA_TRIALS}")
    print(f"{'='*70}")

    for test_year in years:
        # 1. 划分外层训练集与测试集
        train_df = df[df['Year'] != test_year]
        test_df = df[df['Year'] == test_year]
        
        X_train_np = train_df[feature_cols].values
        y_train_np = train_df['yield'].values
        X_test_np = test_df[feature_cols].values
        y_test_np = test_df['yield'].values
        
        # 2. 内层优化：启动 Optuna
        print(f"    [Year {test_year}] 正在寻找年度最优参数...", end="", flush=True)
        study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
        
        func = lambda trial: objective(trial, X_train_np, y_train_np)
        study.optimize(func, n_trials=OPTUNA_TRIALS)
        
        best_params = study.best_params
        # best_params.update({'tree_method': 'hist', 'device': 'cuda', 'random_state': 42, 'n_jobs': -1})
        best_params.update({'tree_method': 'gpu_hist', 'random_state': 42, 'n_jobs': -1})
        print(f" 完成. (最佳深度: {best_params['max_depth']}, colsample: {best_params['colsample_bynode']:.2f})")
        
        # 3. 年度模型定型：使用最佳参数在整个年度训练集上拟合
        X_train_gpu = cp.array(X_train_np)
        y_train_gpu = cp.array(y_train_np)
        X_test_gpu = cp.array(X_test_np)
        
        final_model = XGBRFRegressor(**best_params)
        final_model.fit(X_train_gpu, y_train_gpu)
        
        # 4. 最终评估：预测留出的测试年数据
        y_pred_gpu = cp.asarray(final_model.predict(X_test_gpu))
        y_pred = cp.asnumpy(y_pred_gpu)
        
        all_y_true.extend(y_test_np)
        all_y_pred.extend(y_pred)
        
        # 记录单年结果
        m = calculate_metrics(y_test_np, y_pred)
        m['Test_Year'] = str(test_year)
        m['Best_Params'] = str({k: best_params[k] for k in ['max_depth', 'colsample_bynode', 'subsample']})
        fold_results.append(m)
        
        print(f"    -> [验证结果] R2: {m['R2']:.3f} | RRMSE: {m['RRMSE(%)']:.2f}% | MAPE: {m['MAPE(%)']:.2f}%\n")

    # 5. 计算全局拼接池化指标
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall_Pooled'
    global_metrics['Best_Params'] = 'N/A (Nested)'
    fold_results.append(global_metrics)
    
    # 打印最终成果
    print(f"{'='*70}")
    print(f">>> 嵌套交叉验证全局汇总")
    m = global_metrics
    print(f"    R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}%")
    print(f"    MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
    print(f"{'='*70}")

    # 导出至 CSV
    cols = ['Test_Year', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    results_df = pd.DataFrame(fold_results)[cols]
    results_df.to_csv(output_csv, index=False)
    print(f"学术评估指标已保存至: {output_csv}")

if __name__ == "__main__":
    run_nested_optimization(INPUT_FILE, OUTPUT_CSV)


>>> 启动嵌套 LOYO 交叉验证与超参数优化 (加速版)
    数据集特征维度: 100
    每个外层折叠的 Optuna 寻优次数: 30
    [Year 2016] 正在寻找年度最优参数... 完成. (最佳深度: 12, colsample: 0.47)
    -> [验证结果] R2: 0.175 | RRMSE: 13.90% | MAPE: 9.19%

    [Year 2017] 正在寻找年度最优参数... 完成. (最佳深度: 12, colsample: 0.52)
    -> [验证结果] R2: 0.203 | RRMSE: 13.62% | MAPE: 11.53%

    [Year 2018] 正在寻找年度最优参数... 完成. (最佳深度: 12, colsample: 0.38)
    -> [验证结果] R2: 0.387 | RRMSE: 12.35% | MAPE: 9.60%

    [Year 2019] 正在寻找年度最优参数... 完成. (最佳深度: 12, colsample: 0.42)
    -> [验证结果] R2: 0.365 | RRMSE: 12.48% | MAPE: 9.55%

    [Year 2020] 正在寻找年度最优参数... 完成. (最佳深度: 12, colsample: 0.52)
    -> [验证结果] R2: 0.242 | RRMSE: 11.19% | MAPE: 9.21%

    [Year 2021] 正在寻找年度最优参数... 完成. (最佳深度: 12, colsample: 0.52)
    -> [验证结果] R2: 0.258 | RRMSE: 12.62% | MAPE: 9.69%

>>> 嵌套交叉验证全局汇总
    R2: 0.284 | RMSE: 784.86 | RRMSE: 12.75%
    MAE: 556.80 | MAPE: 9.80% | d-index: 0.698
学术评估指标已保存至: P4_Nested_Metrics.csv
